# IBKR SOXX 期权偏斜前向来源探针

只读检查准确标的、常规到期、期权合约和一次盘后报价。没有历史回测、订单或期权交易。原始连接响应只在本卡内部文件中保留；合约 ID 不在用户结论中显示。

In [ ]:
from pathlib import Path
import json,hashlib,datetime
root=Path.cwd().resolve()
while not (root/"SPEC.md").is_file():
    if root.parent==root:raise RuntimeError("workspace root missing")
    root=root.parent
card=root/"research/ibkr_soxx_option_skew_forward_source_20261022"
pre=json.loads((card/"preflight_plan.json").read_text())
plan=json.loads((card/"sample_plan.json").read_text())
raw_path=card/"market_data_sample.json"
raw=json.loads(raw_path.read_text())
assert pre["frozen"] and plan["frozen"] and pre["budget"]["order_calls"]==plan["budget"]["order_calls"]==0
assert hashlib.sha256((card/"preflight_plan.json").read_bytes()).hexdigest()==plan["preflight_plan_sha256"]
assert len(raw["option_chain"]["contracts"])>0
assert raw["selected_contracts"]["strike"]=="495"
for side in ("call_snapshot","put_snapshot"):
    assert raw[side]["top-status"]["status"]=="FROZEN"
    assert raw[side]["option-midpoint-iv"]["isValid"] is False
    assert raw[side]["option-greeks"]=={}
result={"status":"needs_regular_hours_option_market_data_check","source":"Connected IBKR read-only market-data connector","underlying_quote_status":raw["underlying_snapshot"]["top-status"]["status"],"option_chain_rows":len(raw["option_chain"]["contracts"]),"regular_expiration":plan["expiration"],"selected_strike":raw["selected_contracts"]["strike"],"call_quote_status":raw["call_snapshot"]["top-status"]["status"],"put_quote_status":raw["put_snapshot"]["top-status"]["status"],"call_IV_valid":False,"put_IV_valid":False,"call_Delta_available":False,"put_Delta_available":False,"historical_skew_available":False,"forward_skew_qualified":False,"broker_order_calls":0,"model_fits":0,"accounts":0,"market_data_sample_sha256":hashlib.sha256(raw_path.read_bytes()).hexdigest(),"interpretation":"After-hours FROZEN option quotes with invalid IV and missing Greeks cannot establish an approximately 30-day 25-Delta skew at decision time. Regular-hours sample needed; no absence-of-entitlement inference from this after-hours observation."}
(card/"sample_result.json").write_text(json.dumps(result,indent=2)+chr(10))
summary={"experiment":"ibkr_soxx_option_skew_forward_source_20261022","status":result["status"],"decision":"source_preflight_only_no_skew_factor","result":"sample_result.json","recorder_status":"not_started_source_qualification_only","objective_achieved":False,"qualified_forward_days":0,"model_fits":0,"accounts":0,"artifact_policy":{"policy":"slim_card_recorder_first","workflow_config":"workflow.yaml","recorder_only_artifacts":["sources/*","analysis/*"]}}
(card/"summary.json").write_text(json.dumps(summary,indent=2)+chr(10))
(card/"conclusion.md").write_text("# 中文结论\n\nIBKR 只读连接找到美股 SOXX 期权链和约 29 天后的常规到期，当前链有 "+str(result["option_chain_rows"])+" 个行权价。盘后标的状态为实时，但抽样看涨/看跌期权状态均为 FROZEN，IV 无效、Delta/Greeks 为空。\n\n这只说明盘后样本不能用于固定 25-Delta、约 30 天的偏斜；不能推断白天权限，也不证明历史来源。未创建订单、预警、模型或交易账户。下一步仅在正常交易时间对相同合约做一次预冻结的只读行情核验。\n")
print(result["status"],result["option_chain_rows"])

## 中文结论

盘后样本缺有效 IV/Delta，不能形成固定 Delta 偏斜。详见同卡 `conclusion.md`。